# Capítulo 5 — Programação Orientada a Objetos Aplicada à Engenharia de Software de Defesa

**Programação com Python Aplicada à Engenharia de Defesa** · IPETEC/UCP

> **Notebook de aula — Módulo II, Sábado 2.** Ao final, o miniprojeto integrador deixa de ser um amontoado de *scripts* e passa a ter um **modelo de objetos** claro (conclusão do Módulo 2).

---

### Objetivos do capítulo
Ao final, você será capaz de:
- definir **classes** e criar **objetos**, com atributos e **métodos**;
- aplicar o **encapsulamento** para proteger e validar o estado interno de um objeto;
- reutilizar e especializar código por meio de **herança**;
- escrever código flexível com **polimorfismo**;
- reconhecer **princípios de projeto** que tornam um sistema manutenível e auditável;
- reestruturar o **miniprojeto** em uma arquitetura orientada a objetos.

*Até aqui, dados e funções viviam separados: de um lado, a lista de ocorrências; de outro, as funções que a manipulavam. A **programação orientada a objetos** (POO) reúne, numa única unidade — o objeto —, tanto os dados quanto as operações que lhes dizem respeito. É a passagem de um conjunto de scripts para um sistema de software.*

## 5.1 Classes e objetos

Uma **classe** é um molde — a descrição de um tipo de coisa, com seus atributos (dados) e métodos (comportamentos). Um **objeto** é uma instância concreta desse molde. Se `Ocorrencia` é a classe, uma ocorrência específica, com seu identificador e sua velocidade, é um objeto.

### 5.1.1 Definindo uma classe
A definição começa com `class`. O método especial `__init__` (o **construtor**) é chamado automaticamente ao criar um objeto e inicializa seus atributos. O parâmetro `self` representa o próprio objeto e liga os atributos a ele.

**Listagem 5.1 — Uma primeira classe.**

In [ ]:
class Ocorrencia:
    """Representa uma ocorrência de monitoramento."""

    def __init__(self, id, sensor, velocidade_kmh):
        self.id = id                          # atributos do objeto
        self.sensor = sensor
        self.velocidade_kmh = velocidade_kmh

    def em_alerta(self, limite=40.0):
        """Indica se a velocidade ultrapassa o limite."""
        return self.velocidade_kmh > limite

# Criar objetos é chamar a classe como se fosse uma função
o1 = Ocorrencia(1, "Radar-A1", 44.4)
o2 = Ocorrencia(2, "Sonar-1", 18.5)

print(o1.sensor)         # Radar-A1   (acesso a atributo)
print(o1.em_alerta())    # True       (chamada de método)
print(o2.em_alerta())    # False

A classe `Ocorrencia` é escrita **uma vez**; a partir dela, criamos quantos objetos quisermos, cada um com seus próprios valores. O método `em_alerta` é chamado sobre um objeto específico (`o1.em_alerta()`) e opera sobre os dados *daquele* objeto, acessados via `self`.

> 📝 **Nota** — O `self` aparece como primeiro parâmetro na **definição** de todo método, mas não é passado na **chamada**: ao escrever `o1.em_alerta()`, o Python entende `o1` como o `self`. Esquecer o `self` na definição de um método é um dos enganos mais comuns de quem começa em POO.

### 5.1.2 Representação legível de um objeto
Por padrão, ao imprimir um objeto, o Python exibe algo pouco informativo, como `<__main__.Ocorrencia object at 0x7f...>`. Veja você mesmo:

In [ ]:
class SemRepr:
    def __init__(self, valor):
        self.valor = valor

print(SemRepr(42))     # algo como <__main__.SemRepr object at 0x...>

O método especial `__repr__` permite definir uma representação textual útil — valiosa para depuração e para registros (*logs*) auditáveis.

**Listagem 5.2 — Definindo uma representação legível com `__repr__`.**

In [ ]:
class Ocorrencia:
    def __init__(self, id, sensor, velocidade_kmh):
        self.id = id
        self.sensor = sensor
        self.velocidade_kmh = velocidade_kmh

    def __repr__(self):
        return (f"Ocorrencia(#{self.id}, {self.sensor}, "
                f"{self.velocidade_kmh} km/h)")

o = Ocorrencia(1, "Radar-A1", 44.4)
print(o)         # Ocorrencia(#1, Radar-A1, 44.4 km/h)

## 5.2 Encapsulamento

Um dos princípios centrais da POO é o **encapsulamento**: proteger o estado interno de um objeto, controlando como ele pode ser lido e alterado. Em vez de expor os atributos livremente, a classe oferece uma interface que pode **validar** as mudanças — garantindo que o objeto nunca caia em um estado inválido.

Em Python, a convenção é prefixar com um sublinhado (`_velocidade`) os atributos internos. Para controlar o acesso, usa-se o decorador `@property`, que transforma um método em algo que se acessa como um atributo, mas com a possibilidade de validação.

**Listagem 5.3 — Encapsulamento com property e validação.**

In [ ]:
class Ocorrencia:
    def __init__(self, sensor, velocidade_kmh):
        self.sensor = sensor
        self.velocidade_kmh = velocidade_kmh   # já usa o setter abaixo

    @property
    def velocidade_kmh(self):
        """Leitura do atributo protegido."""
        return self._velocidade_kmh

    @velocidade_kmh.setter
    def velocidade_kmh(self, valor):
        """Escrita com validação."""
        if valor < 0:
            raise ValueError("A velocidade não pode ser negativa.")
        self._velocidade_kmh = valor

o = Ocorrencia("Radar-A1", 44.4)
print(o.velocidade_kmh)      # 44.4  (lê como atributo)

o.velocidade_kmh = 50.0      # atribui como atributo, mas valida
print(o.velocidade_kmh)      # 50.0

De fora, `o.velocidade_kmh` continua parecendo um atributo comum — mas qualquer tentativa de atribuir um valor inválido é barrada na origem. Comprovemos que o *setter* realmente recusa um valor negativo:

**Listagem 5.3b — O setter em ação: a validação barra o valor inválido.**

In [ ]:
try:
    o.velocidade_kmh = -3        # tentativa inválida
except ValueError as erro:
    print("Recusado pelo setter:", erro)

print("Valor preservado:", o.velocidade_kmh)   # continua 50.0

> 🛡️ **Contexto de defesa** — Em sistemas auditáveis — e os de defesa quase sempre o são —, o encapsulamento vai além da organização. Quando um objeto garante, **por construção**, que nunca assume um estado inválido, elimina-se uma classe inteira de erros e reduz-se a superfície de auditoria: não é preciso vasculhar todo o código em busca de quem poderia ter atribuído um valor incorreto, pois apenas um ponto — o *setter* — controla a escrita.

## 5.3 Herança

A **herança** permite que uma classe (a *subclasse*) reaproveite e especialize outra (a *superclasse*). É a forma de exprimir, em código, uma relação do tipo "é um(a)": um radar *é um* sensor; um sonar *é um* sensor. O que há de comum vive na superclasse; o que é específico, nas subclasses.

**Listagem 5.4 — Uma hierarquia de sensores.**

In [ ]:
class Sensor:
    def __init__(self, nome, alcance_km):
        self.nome = nome
        self.alcance_km = alcance_km

    def descricao(self):
        return f"{self.nome}, alcance {self.alcance_km} km"

class Radar(Sensor):
    def __init__(self, nome, alcance_km, frequencia_ghz):
        super().__init__(nome, alcance_km)    # chama o construtor da base
        self.frequencia_ghz = frequencia_ghz

class Sonar(Sensor):
    def __init__(self, nome, alcance_km, profundidade_max):
        super().__init__(nome, alcance_km)
        self.profundidade_max = profundidade_max

radar = Radar("Radar-A1", 120, 9.4)
print(radar.descricao())        # herdado de Sensor
print(radar.frequencia_ghz)     # específico de Radar

sonar = Sonar("Sonar-1", 30, 500)
print(sonar.descricao())        # o mesmo método herdado serve ao Sonar

A função `super()` é a chave: dá acesso à superclasse. No construtor de `Radar`, `super().__init__(nome, alcance_km)` aproveita a inicialização já escrita em `Sensor`, evitando repetição. Cada subclasse acrescenta apenas o que lhe é próprio. O método `descricao`, definido só na superclasse, fica disponível para todas as subclasses.

> ✅ **Boa prática** — Use herança para modelar relações genuínas de "é um(a)". Se você se vê herdando apenas para reaproveitar algumas linhas, sem que a subclasse seja realmente um tipo da superclasse, talvez o melhor seja **composição** — um objeto conter outro como atributo. A pergunta-guia é simples: "um Radar *é um* Sensor?" Se sim, a herança é adequada.

## 5.4 Polimorfismo

**Polimorfismo** — do grego, "muitas formas" — é a capacidade de objetos de classes diferentes responderem ao *mesmo* método, cada um à sua maneira. Uma subclasse pode **sobrescrever** um método da superclasse, oferecendo sua própria implementação. Quem chama o método não precisa saber de que tipo é o objeto: o Python escolhe a implementação correta.

**Listagem 5.5 — Polimorfismo: o mesmo método, comportamentos distintos.**

In [ ]:
class Sensor:
    def __init__(self, nome):
        self.nome = nome

    def tipo(self):
        return "sensor genérico"

class Radar(Sensor):
    def tipo(self):                      # sobrescreve o método da base
        return "radar (detecção eletromagnética)"

class Sonar(Sensor):
    def tipo(self):
        return "sonar (detecção acústica)"

# Uma lista de sensores de tipos diferentes
sensores = [Radar("Radar-A1"), Sonar("Sonar-1"), Sensor("Padrão")]

# O mesmo laço trata todos, e cada um responde à sua maneira
for s in sensores:
    print(f"{s.nome}: {s.tipo()}")

O laço não contém um único `if` para distinguir tipos de sensor. Cada objeto "sabe" responder a `tipo()`, e o resultado correto emerge naturalmente. Adicionar um novo tipo — digamos, um `Lidar` — não exige alterar o laço: basta criar a nova subclasse. Essa é a grande virtude do polimorfismo: o código que *usa* os objetos permanece estável enquanto o conjunto de tipos cresce.

> 🛡️ **Contexto de defesa** — O polimorfismo é o que permite que um sistema de comando e controle trate, de forma uniforme, contatos heterogêneos — de superfície, aéreos, submarinos — sem um emaranhado de condicionais. Cada tipo de contato encapsula seu próprio comportamento; o sistema apenas pede "classifique-se", "represente-se no mapa", "calcule sua ameaça", e cada objeto responde adequadamente. O código cresce por **adição** de classes, não por **modificação** do que já funciona.

## 5.5 Princípios de projeto para sistemas manuteníveis

A POO oferece as ferramentas; usá-las bem é uma questão de projeto. Alguns princípios, simples de enunciar e valiosos na prática, orientam a construção de sistemas que se mantêm saudáveis ao longo do tempo — atributo essencial em software de defesa, de ciclo de vida longo e exigências de auditoria.

- **Responsabilidade única** — cada classe deve ter uma única razão para existir e mudar. `Ocorrencia` cuida de uma ocorrência; quem gerencia a coleção e a persistência é outra classe.
- **Encapsulamento do estado** — exponha o mínimo necessário. Quanto menos o mundo externo puder alterar diretamente, menos caminhos há para um estado inválido.
- **Coesão e baixo acoplamento** — mantenha junto o que muda junto (coesão) e reduza as dependências entre partes (acoplamento).
- **Nomes que revelam intenção** — `em_alerta`, `registrar`: um nome claro vale mais que qualquer comentário, sobretudo em código que auditores e outros engenheiros lerão anos depois.

> 📝 **Nota** — Esses princípios não são regras rígidas, mas heurísticas. Aplicá-los com bom senso — e não dogmaticamente — é parte da maturidade do engenheiro de software. Um sistema pequeno não precisa da mesma cerimônia de um grande.

## 5.6 Miniprojeto: reestruturação em objetos (Módulo 2)

Reunimos agora tudo do Módulo 2 em uma arquitetura orientada a objetos. Duas classes dividem as responsabilidades: `Ocorrencia` representa uma ocorrência individual — e sabe converter-se de e para dicionário, o que viabiliza a persistência em JSON do capítulo anterior; `RegistroDeOcorrencias` gerencia a coleção e a persistência. É a aplicação direta do princípio da **responsabilidade única**.

**Listagem 5.6 — Miniprojeto, Módulo 2 (conclusão): arquitetura orientada a objetos.**

In [ ]:
import json

class Ocorrencia:
    """Uma ocorrência de monitoramento."""

    def __init__(self, id, sensor, velocidade_kmh, tipo="superfície"):
        self.id = id
        self.sensor = sensor
        self.velocidade_kmh = velocidade_kmh
        self.tipo = tipo

    def em_alerta(self, limite=40.0):
        return self.velocidade_kmh > limite

    def para_dict(self):
        """Converte o objeto em dicionário (para o JSON)."""
        return {
            "id": self.id, "sensor": self.sensor,
            "velocidade_kmh": self.velocidade_kmh, "tipo": self.tipo,
        }

    @classmethod
    def de_dict(cls, d):
        """Cria uma Ocorrencia a partir de um dicionário."""
        return cls(d["id"], d["sensor"], d["velocidade_kmh"],
                   d.get("tipo", "superfície"))

    def __repr__(self):
        return f"Ocorrencia(#{self.id}, {self.sensor}, {self.velocidade_kmh} km/h)"


class RegistroDeOcorrencias:
    """Gerencia a coleção de ocorrências e a sua persistência."""

    def __init__(self, limite=40.0):
        self._ocorrencias = []
        self.limite = limite

    def registrar(self, sensor, velocidade_kmh, tipo="superfície"):
        novo_id = len(self._ocorrencias) + 1
        ocorrencia = Ocorrencia(novo_id, sensor, velocidade_kmh, tipo)
        self._ocorrencias.append(ocorrencia)
        return ocorrencia

    def alertas(self):
        return [o for o in self._ocorrencias if o.em_alerta(self.limite)]

    def total(self):
        return len(self._ocorrencias)

    def salvar(self, caminho):
        dados = [o.para_dict() for o in self._ocorrencias]
        with open(caminho, "w", encoding="utf-8") as f:
            json.dump(dados, f, ensure_ascii=False, indent=2)

    def carregar(self, caminho):
        try:
            with open(caminho, "r", encoding="utf-8") as f:
                dados = json.load(f)
            self._ocorrencias = [Ocorrencia.de_dict(d) for d in dados]
        except FileNotFoundError:
            self._ocorrencias = []


# ---- Uso ----
registro = RegistroDeOcorrencias(limite=40.0)
registro.registrar("Radar-A1", 44.4)
registro.registrar("Sonar-1", 18.5, tipo="submarino")
registro.registrar("Radar-B2", 50.0, tipo="aéreo")

print(f"Total de ocorrências: {registro.total()}")
print("Em alerta:")
for o in registro.alertas():
    print(f"  {o}")

registro.salvar("ocorrencias.json")

Compare este código com a versão do Capítulo 3, feita de dicionários soltos e funções avulsas. Agora, cada ocorrência é um objeto coeso, que sabe dizer se está em alerta e converter-se para persistência; o registro é um objeto que centraliza a coleção e as operações sobre ela. O método de classe `de_dict` — marcado com `@classmethod` — é uma **fábrica**: constrói uma `Ocorrencia` a partir dos dados lidos do arquivo, fechando o ciclo entre os objetos e o JSON do capítulo anterior.

Vejamos esse ciclo se fechar na prática: um **novo** registro, partindo vazio, recupera do arquivo os objetos que acabamos de salvar.

**Listagem 5.6b — Recuperando os objetos do disco (a fábrica `de_dict` em ação).**

In [ ]:
# Um registro totalmente novo, que nada sabe das ocorrências anteriores
outro = RegistroDeOcorrencias()
outro.carregar("ocorrencias.json")

print(f"Recuperadas {outro.total()} ocorrências do disco.")
for o in outro.alertas():
    print(f"  {o}")            # note: já são objetos Ocorrencia, não dicionários

primeira = outro._ocorrencias[0]
print(type(primeira).__name__, "->", primeira.em_alerta())   # Ocorrencia -> True

> 📝 **Nota** — Com isto, encerramos o **Módulo 2**. O miniprojeto deixou de ser um amontoado de *scripts*: tem agora um modelo de objetos claro, persistência confiável e fronteiras de responsabilidade bem definidas. É uma base sólida sobre a qual os próximos módulos acrescentarão uma interface gráfica (Módulo 3) e a análise de dados (Módulo 4).

> 💡 **No Colab** — A célula da Listagem 5.6 cria o arquivo `ocorrencias.json` na pasta `/content` (ícone de pasta, à esquerda). Ele é temporário: some ao encerrar a sessão. Execute as células **na ordem**, pois a 5.6b lê o que a 5.6 gravou.

## 5.7 O caminho à frente

Com o miniprojeto reorganizado em objetos, fecha-se o segundo módulo do curso. O **Módulo 3** muda o foco do *dado* para a *interação* e o *cálculo*: o Capítulo 6 ensina a automatizar tarefas e a construir uma interface gráfica com **Tkinter**, pela qual um operador poderá usar o sistema sem tocar no código; e o Capítulo 7 apresenta o **NumPy**, para os cálculos numéricos que sustentam a análise. O sistema começará, enfim, a ganhar a forma de uma aplicação que se entrega a um usuário.

## 5.8 Resumo do capítulo
- Uma **classe** é um molde com atributos e métodos; um **objeto** é uma instância dela. O construtor `__init__` inicializa os atributos, e `self` representa o próprio objeto.
- O **encapsulamento** protege o estado interno: a convenção do sublinhado (`_atributo`) e o decorador `@property` permitem **validar** mudanças, garantindo que o objeto nunca fique inválido.
- A **herança** (`class Sub(Base)`) reaproveita e especializa código, modelando relações "é um(a)"; `super()` acessa a superclasse.
- O **polimorfismo** permite que objetos de classes diferentes respondam ao mesmo método à sua maneira, eliminando condicionais e favorecendo o crescimento por *adição* de classes.
- **Princípios de projeto** — responsabilidade única, encapsulamento, coesão e baixo acoplamento, nomes que revelam intenção — guiam sistemas manuteníveis e auditáveis.
- No **Módulo 2 concluído**, o miniprojeto foi reestruturado em duas classes coesas, `Ocorrencia` e `RegistroDeOcorrencias`, unindo modelo de objetos e persistência.

## Armadilhas comuns
- **Esquecer o `self`.** Todo método de instância recebe `self` como primeiro parâmetro na definição; omiti-lo provoca erro na primeira chamada.
- **Confundir classe com objeto.** A classe é o molde, escrito uma vez; o objeto é a instância concreta, criada ao chamar a classe.
- **Acessar atributos protegidos diretamente.** Respeite a convenção do sublinhado; altere o estado pela interface da classe (*property* ou método).
- **Esquecer `super().__init__()`.** Numa subclasse com construtor próprio, deixar de chamar o construtor da superclasse costuma deixar atributos herdados sem inicializar.
- **Herdar sem relação "é um(a)".** Herança só para reaproveitar linhas gera hierarquias confusas; considere composição.
- **Criar classes que fazem tudo.** Uma classe com responsabilidades demais é difícil de testar e manter; divida-as.

## Exercícios

### Essencial — fixação
**Ex. 5.1** Crie uma classe `Meio` com os atributos `nome` e `velocidade_max_kmh`, inicializados no construtor. Adicione um método `descricao` que devolva uma frase com os dois dados. Crie dois objetos e imprima suas descrições.

In [ ]:
# Ex. 5.1
class Meio:
    def __init__(self, nome, velocidade_max_kmh):
        self.nome = nome
        self.velocidade_max_kmh = velocidade_max_kmh

    def descricao(self):
        return f"{self.nome}: velocidade máxima de {self.velocidade_max_kmh} km/h"

fragata = Meio("Fragata", 56.0)
corveta = Meio("Corveta", 74.0)
print(fragata.descricao())
print(corveta.descricao())

**Ex. 5.2** Acrescente à classe `Meio` um método `__repr__` que produza uma representação legível, no formato `Meio(nome, vel_max)`. Crie um objeto e imprima-o diretamente com `print`.

In [ ]:
# Ex. 5.2
class Meio:
    def __init__(self, nome, velocidade_max_kmh):
        self.nome = nome
        self.velocidade_max_kmh = velocidade_max_kmh

    def descricao(self):
        return f"{self.nome}: velocidade máxima de {self.velocidade_max_kmh} km/h"

    def __repr__(self):
        return f"Meio({self.nome}, {self.velocidade_max_kmh})"

print(Meio("Fragata", 56.0))     # Meio(Fragata, 56.0)

**Ex. 5.3** Escreva uma classe `Sensor` com os atributos `nome` e `ativo` (lógico). Adicione os métodos `ligar` e `desligar`, que alteram `ativo` para `True` e `False`, e um método `status` que informe o estado atual.

In [ ]:
# Ex. 5.3
class Sensor:
    def __init__(self, nome, ativo=False):
        self.nome = nome
        self.ativo = ativo

    def ligar(self):
        self.ativo = True

    def desligar(self):
        self.ativo = False

    def status(self):
        return f"{self.nome}: {'ligado' if self.ativo else 'desligado'}"

s = Sensor("Radar-A1")
print(s.status())      # desligado
s.ligar()
print(s.status())      # ligado

### Tático — aplicação
**Ex. 5.4** Reescreva a classe `Meio` aplicando **encapsulamento**: torne `velocidade_max_kmh` um atributo protegido, acessível por uma `@property`, cujo *setter* recuse valores negativos lançando `ValueError`.

In [ ]:
# Ex. 5.4
class Meio:
    def __init__(self, nome, velocidade_max_kmh):
        self.nome = nome
        self.velocidade_max_kmh = velocidade_max_kmh   # usa o setter

    @property
    def velocidade_max_kmh(self):
        return self._velocidade_max_kmh

    @velocidade_max_kmh.setter
    def velocidade_max_kmh(self, valor):
        if valor < 0:
            raise ValueError("A velocidade máxima não pode ser negativa.")
        self._velocidade_max_kmh = valor

m = Meio("Fragata", 56.0)
print(m.velocidade_max_kmh)      # 56.0

try:
    m.velocidade_max_kmh = -10
except ValueError as erro:
    print("Recusado:", erro)

**Ex. 5.5** Crie uma superclasse `Meio` e duas subclasses, `Navio` e `Aeronave`, cada uma com um atributo próprio (por exemplo, `calado_m` para o navio e `teto_servico_m` para a aeronave). Use `super()` no construtor de cada subclasse.

In [ ]:
# Ex. 5.5
class Meio:
    def __init__(self, nome, velocidade_max_kmh):
        self.nome = nome
        self.velocidade_max_kmh = velocidade_max_kmh

class Navio(Meio):
    def __init__(self, nome, velocidade_max_kmh, calado_m):
        super().__init__(nome, velocidade_max_kmh)
        self.calado_m = calado_m

class Aeronave(Meio):
    def __init__(self, nome, velocidade_max_kmh, teto_servico_m):
        super().__init__(nome, velocidade_max_kmh)
        self.teto_servico_m = teto_servico_m

fragata = Navio("Fragata", 56.0, calado_m=4.5)
caca = Aeronave("Caça", 2200.0, teto_servico_m=15000)
print(fragata.nome, "- calado", fragata.calado_m, "m")
print(caca.nome, "- teto de serviço", caca.teto_servico_m, "m")

**Ex. 5.6** Dê às classes `Navio` e `Aeronave` um método `dominio` que devolva, respectivamente, `"superfície"` e `"ar"`. Crie uma lista com meios dos dois tipos e, em um único laço, imprima o domínio de cada um — demonstrando **polimorfismo**.

In [ ]:
# Ex. 5.6
class Meio:
    def __init__(self, nome, velocidade_max_kmh):
        self.nome = nome
        self.velocidade_max_kmh = velocidade_max_kmh

    def dominio(self):
        return "indefinido"

class Navio(Meio):
    def dominio(self):
        return "superfície"

class Aeronave(Meio):
    def dominio(self):
        return "ar"

meios = [Navio("Fragata", 56.0), Aeronave("Caça", 2200.0)]
for m in meios:               # um único laço, sem 'if' de tipo
    print(f"{m.nome}: domínio {m.dominio()}")

### Estratégico — extensão criativa
**Ex. 5.7** Estenda a classe `RegistroDeOcorrencias` da Listagem 5.6 com um método `resumo_por_tipo` (dicionário com a contagem por tipo) e um método `velocidade_media` (média das velocidades registradas). Reflita: esses métodos pertencem naturalmente ao **registro**, não à ocorrência individual.

*(Reutiliza `Ocorrencia` e `RegistroDeOcorrencias` da Listagem 5.6 — execute aquela célula antes.)*

In [ ]:
# Ex. 5.7
class RegistroDeOcorrenciasPlus(RegistroDeOcorrencias):

    def resumo_por_tipo(self):
        contagem = {}
        for o in self._ocorrencias:
            contagem[o.tipo] = contagem.get(o.tipo, 0) + 1
        return contagem

    def velocidade_media(self):
        if not self._ocorrencias:
            return 0.0
        soma = sum(o.velocidade_kmh for o in self._ocorrencias)
        return soma / len(self._ocorrencias)

reg = RegistroDeOcorrenciasPlus()
reg.registrar("Radar-A1", 44.4, tipo="superfície")
reg.registrar("Sonar-1", 18.5, tipo="submarino")
reg.registrar("Radar-B2", 50.0, tipo="aéreo")
reg.registrar("Radar-A1", 30.0, tipo="superfície")

print("Resumo por tipo:", reg.resumo_por_tipo())
print(f"Velocidade média: {reg.velocidade_media():.1f} km/h")

**Ex. 5.8** Projete **apenas o esqueleto** (classes com métodos e *docstrings*, sem implementação completa) de uma pequena hierarquia para representar tipos de `Contato` em um sistema de comando e controle: `ContatoSuperficie`, `ContatoAereo` e `ContatoSubmarino`, derivados de `Contato`. Decida quais atributos e comportamentos pertencem à superclasse e quais são específicos de cada subclasse, justificando à luz dos princípios da Seção 5.5.

In [ ]:
# Ex. 5.8 — esqueleto de projeto (use 'pass' onde não implementar)
class Contato:
    """Contato genérico. Reúne o que é COMUM a todo contato:
    identificação, posição e velocidade — atributos que todo contato tem,
    independentemente do domínio (responsabilidade única na superclasse)."""

    def __init__(self, id, posicao, velocidade_kmh):
        self.id = id
        self.posicao = posicao              # tupla (lat, lon)
        self.velocidade_kmh = velocidade_kmh

    def classificar(self):
        """Classifica o nível de ameaça. Cada subclasse sobrescreve
        com o critério do seu domínio (ponto de extensão polimórfico)."""
        raise NotImplementedError

    def dominio(self):
        """Devolve o domínio do contato (a ser definido pela subclasse)."""
        raise NotImplementedError


class ContatoSuperficie(Contato):
    """Contato de superfície. Específico: calado, tipo de casco..."""
    def dominio(self):
        return "superfície"
    def classificar(self):
        pass   # critério de ameaça de superfície


class ContatoAereo(Contato):
    """Contato aéreo. Específico: altitude, rota de voo..."""
    def dominio(self):
        return "ar"
    def classificar(self):
        pass   # critério de ameaça aérea


class ContatoSubmarino(Contato):
    """Contato submarino. Específico: profundidade, assinatura acústica..."""
    def dominio(self):
        return "submarino"
    def classificar(self):
        pass   # critério de ameaça submarina


# Demonstração do esqueleto (o polimorfismo já funciona para 'dominio'):
for c in [ContatoSuperficie(1, (-22.9, -43.2), 40.0),
          ContatoAereo(2, (-22.7, -43.0), 900.0),
          ContatoSubmarino(3, (-23.0, -43.3), 30.0)]:
    print(f"Contato #{c.id}: domínio {c.dominio()}")

---

*Fim do Capítulo 5 e do Módulo 2. No Capítulo 6, o sistema ganha uma **interface gráfica** (Tkinter) — a passagem do código para uma aplicação de verdade.*